# Notebook 17 — Mini Assessment (Answer Key)
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers

Every conceptual question answered in full, and every coding exercise implemented and
executed against the real Telco Customer Churn dataset — matching the same fully-worked
format used for Sprint 4's assessment.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"Dataset ready: {df.shape[0]:,} rows")


Dataset ready: 7,043 rows


---
## Conceptual Questions (Answered)


### 1. What is data preprocessing?

The set of steps that convert raw, as-collected data into a clean, consistent, and
correctly-structured form a machine learning algorithm can actually use — everything
between "loading a dataset" and "handing it to a model." In this sprint, that meant fixing
`TotalCharges`'s data type, resolving its missing values, validating the result,
deciding on outlier treatment, encoding categoricals, scaling numerics, and assembling it
all into a leakage-safe pipeline.


### 2. Why is preprocessing required?

Because a model literally cannot compensate for poor-quality data. Demonstrated directly
in Notebook 1: attempting to train `LogisticRegression` on the raw, unconverted
`TotalCharges` column raises a `ValueError` immediately — preprocessing isn't optional
polish, it's a hard prerequisite for the model to run at all, let alone run well.


### 3. Explain MCAR, MAR, and MNAR.

- **MCAR** (Missing Completely At Random): missingness has no relationship to any
  variable at all — pure chance.
- **MAR** (Missing At Random): missingness relates to an *observed* variable, but not to
  the missing value itself. `TotalCharges`'s 11 gaps are a textbook MAR case — 100% of
  them occur exactly where `tenure == 0`.
- **MNAR** (Missing Not At Random): missingness relates to the unobserved value itself
  (e.g., very high earners being less likely to report income).

Classifying `TotalCharges` as MAR (Notebook 3) is exactly what justified filling with a
constant 0 instead of mean/median imputation.


### 4. How would you handle 40% missing values?

At 40% missing, imputation becomes increasingly speculative — that much of a column is
being *invented*, not estimated. The right response depends on the missingness type
(MCAR/MAR/MNAR) and the column's importance: if MCAR and the column is not critical,
dropping the column may be more honest than imputing nearly half its values; if MAR and a
strong predictor of the gap exists, a targeted group-based or model-based imputation
(KNN/Iterative, Notebook 3) can be defensible; if the missingness itself carries signal
(common in MNAR), engineering a "was this value missing" binary flag alongside any
imputation is often more useful than the imputed value itself. There's no universal
answer at 40% — it demands more scrutiny, not a bigger hammer.


### 5. Mean vs. Median Imputation.

Both fill gaps with a single central-tendency statistic. **Mean** is appropriate for
roughly symmetric, MCAR numeric data — it's pulled toward any skew or outliers, though.
**Median** is more robust to skew and outliers, since it's rank-based, not
magnitude-based. In Notebook 3, BOTH were rejected for `TotalCharges` — not because one
is inherently better, but because neither respects the MAR relationship to `tenure`;
either would assign a nonsensical non-zero bill to brand-new, not-yet-billed customers.
This is the core lesson: the choice between mean and median matters less than first
asking whether a central-tendency fill is appropriate at all.


### 6. What is data leakage?

Information that would NOT actually be available at real prediction time accidentally
influencing model training — producing misleadingly high performance during development
that collapses in real deployment. Demonstrated numerically in Notebook 13: a feature
that only exists *because* a customer already churned pushed test accuracy to
near-perfect, purely because it was circular, not because the model learned anything
real.


### 7. Explain target leakage.

A specific type of data leakage where a feature is a direct byproduct of the target
itself. Notebook 13's example: `had_retention_call`, engineered as `1` only when
`Churn == 'Yes'`. A model trained on this feature would never actually have access to it
for a real, not-yet-churned customer — it's the answer, disguised as a question.


### 8. What is one-hot encoding?

Creates one new binary (0/1) column per category, with no implied numeric order between
them. In Notebook 7, this was the correct choice for every genuinely nominal column
(`InternetService`, `PaymentMethod`, etc.) — as opposed to `Contract`, which got ordinal
encoding instead, since it has a genuine order (Month-to-month < One year < Two year).


### 9. Label Encoding vs. One-Hot Encoding.

**Label encoding** assigns each category an arbitrary integer — fine for a binary column
(only 2 categories, so no false order is implied) or an ordinal column (where the order
IS meaningful), but risky for a nominal column with 3+ categories, since it implies a
numeric distance/order that doesn't exist. **One-hot encoding** avoids that risk entirely
by creating a separate binary column per category, at the cost of more total columns.
Notebook 7 demonstrated the risk directly: label-encoding `InternetService` would let a
linear model treat "No"=2 as "twice" Fiber optic=1 — meaningless — which is exactly why
one-hot was used for it instead.


### 10. Standardization vs. Normalization.

**Standardization** (`StandardScaler`) rescales to mean=0, std=1, with no fixed
boundaries. **Normalization** (commonly `MinMaxScaler`) rescales to a fixed range, usually
[0,1]. Neither fixes skew — that's a transformation's job (Notebook 9), not a scaler's.
Notebook 8 selected `StandardScaler` for this dataset specifically because its numeric
columns are confirmed outlier-free (Notebook 6) — there was no reason to reach for a more
defensive option.


### 11. When would you use RobustScaler?

When a numeric column has real outliers (or their presence is uncertain). `RobustScaler`
uses median and IQR instead of mean/std, so it isn't distorted by extreme values the way
`StandardScaler` and especially `MinMaxScaler` are. Notebook 8 demonstrated this directly
by injecting one synthetic $500,000 outlier: `MinMaxScaler` crushed every normal
customer's scaled value into a tiny sliver near 0, while `RobustScaler` was barely
affected. For THIS dataset's actual (outlier-free) columns, `StandardScaler` was
selected instead — `RobustScaler` would have been equally valid but offered no extra
benefit here.


### 12. What is class imbalance?

When one target class substantially outnumbers another. This dataset's `Churn` splits
73.46% No / 26.54% Yes — a 2.77:1 ratio, classified as moderate (Notebook 11), not
severe. The core danger: a model can score deceptively high accuracy just by favoring the
majority class — demonstrated directly with an "always predict No" model scoring 73.4%
accuracy while catching zero actual churners.


### 13. Explain SMOTE.

Synthetic Minority Oversampling Technique — instead of duplicating minority-class rows
exactly (which risks overfitting to repeated examples), SMOTE generates NEW synthetic
minority examples by interpolating between real minority neighbors in feature space.
Notebook 11 confirmed the synthetic rows are not exact duplicates of any real row, and
compared SMOTE's precision/recall/F1 trade-off directly against undersampling,
oversampling, Borderline-SMOTE, and class weighting on the same held-out test set.


### 14. Why should preprocessing be fitted only on training data?

Because fitting on the full dataset (or the test set) lets information that should be
"unseen" leak into the model, producing an overoptimistic performance estimate that won't
hold up in real deployment. Notebook 12 demonstrated this numerically: a `StandardScaler`
fit on the full dataset learns a measurably different mean than one fit correctly on the
training fold alone — real, quantified train-test contamination, not a hypothetical risk.


### 15. What is a preprocessing pipeline?

A single object (`sklearn.pipeline.Pipeline`) that chains preprocessing steps and a final
model together, fit and used as one unit. It's not just convenience — it structurally
*enforces* the fit-on-train discipline (Q14), since calling `.fit()` once on training
data correctly scopes every internal step, making it far harder to accidentally leak
information the way manual, step-by-step code could (Notebook 14).


### 16. Why use `ColumnTransformer`?

Because different column types need entirely different preprocessing — numeric columns
need imputation and scaling; categorical columns need imputation and encoding.
`ColumnTransformer` applies a different `Pipeline` to each column group in parallel, then
combines the results into one output matrix — exactly what Notebook 14's final pipeline
does, with a numeric sub-pipeline (constant-fill → `StandardScaler`) and a categorical
sub-pipeline (mode-fill → `OneHotEncoder`) running side by side.


---
## Coding Exercises (Implemented & Executed)


### Exercise 1. Detect and treat outliers.

In [2]:
from scipy import stats
df_ex = df.copy()
df_ex['TotalCharges'] = df_ex['TotalCharges'].fillna(0)
df_ex['expected_total'] = df_ex['tenure'] * df_ex['MonthlyCharges']
df_ex['residual'] = df_ex['TotalCharges'] - df_ex['expected_total']

outliers = df_ex[np.abs(stats.zscore(df_ex['residual'])) > 3]
print(f"Multivariate outliers detected: {len(outliers)}")
print("Treatment decision: RETAIN (genuine billing-history variation, not errors) — see Notebook 6 for full reasoning.")


Multivariate outliers detected: 112
Treatment decision: RETAIN (genuine billing-history variation, not errors) — see Notebook 6 for full reasoning.


### Exercise 2. Encode categorical features.

In [3]:
ohe = OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)
encoded = ohe.fit_transform(df_ex[['InternetService', 'PaymentMethod']])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(['InternetService', 'PaymentMethod']))
print(f"Original columns: 2 -> Encoded columns: {encoded_df.shape[1]}")
print(encoded_df.head(3))


Original columns: 2 -> Encoded columns: 5
   InternetService_Fiber optic  InternetService_No  \
0                          0.0                 0.0   
1                          0.0                 0.0   
2                          0.0                 0.0   

   PaymentMethod_Credit card (automatic)  PaymentMethod_Electronic check  \
0                                    0.0                             1.0   
1                                    0.0                             0.0   
2                                    0.0                             0.0   

   PaymentMethod_Mailed check  
0                         0.0  
1                         1.0  
2                         1.0  


### Exercise 3. Build a complete preprocessing pipeline.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = ['gender', 'Contract', 'InternetService', 'PaymentMethod']

numeric_pipe = Pipeline([('impute', SimpleImputer(strategy='constant', fill_value=0)), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                              ('encode', OneHotEncoder(handle_unknown='ignore', drop='first'))])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols),
    ('cat', categorical_pipe, categorical_cols),
])

full_pipeline = Pipeline([('prep', preprocessor), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])

X = df_ex[numeric_cols + categorical_cols]
y = (df_ex['Churn'] == 'Yes').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

full_pipeline.fit(X_train, y_train)
print(f"Pipeline built and fit successfully. Test accuracy: {full_pipeline.score(X_test, y_test):.4f}")


Pipeline built and fit successfully. Test accuracy: 0.7324


---
## Summary

Every question above ties back to a specific, re-verified finding from this sprint's 16
preceding notebooks — not a generic textbook answer. This is meant as a study reference:
being able to reconstruct *why* each answer is correct (not just recall it) is exactly
what a review session tests.

**This completes Sprint 5 — Data Cleaning & Preprocessing.** The next sprint (Feature
Engineering) builds directly on the ML-ready dataset (`telco_churn_ml_ready.csv`) and
pipeline produced in Notebook 16.
